# 🟡 Solution: Batch Segment Intersection

**Primitive:** broadcasting cross-product sign test

**Reduction:** `out[i,j]` is True iff d1·d2 < 0 AND d3·d4 < 0 where d1..d4 are the four orientation cross products, broadcast over all (N,M) pairs simultaneously.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: broadcasting cross-product orientation test

import numpy as np

def segment_intersect(segs_a, segs_b):
    segs_a = np.asarray(segs_a, dtype=float)
    segs_b = np.asarray(segs_b, dtype=float)
    # Broadcast (N,1,2) vs (1,M,2)
    A = segs_a[:, None, 0, :]; B = segs_a[:, None, 1, :]
    C = segs_b[None, :, 0, :]; D = segs_b[None, :, 1, :]

    def cross2d(u, v):
        return u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]

    d1 = cross2d(B-A, C-A); d2 = cross2d(B-A, D-A)
    d3 = cross2d(D-C, A-C); d4 = cross2d(D-C, B-C)
    proper = (d1*d2 < 0) & (d3*d4 < 0)

    def on_seg(px, py, ax, ay, bx, by):
        return ((np.minimum(ax,bx) <= px) & (px <= np.maximum(ax,bx)) &
                (np.minimum(ay,by) <= py) & (py <= np.maximum(ay,by)))

    Ax,Ay=A[...,0],A[...,1]; Bx,By=B[...,0],B[...,1]
    Cx,Cy=C[...,0],C[...,1]; Dx,Dy=D[...,0],D[...,1]
    col = (((d1==0) & on_seg(Cx,Cy,Ax,Ay,Bx,By)) |
           ((d2==0) & on_seg(Dx,Dy,Ax,Ay,Bx,By)) |
           ((d3==0) & on_seg(Ax,Ay,Cx,Cy,Dx,Dy)) |
           ((d4==0) & on_seg(Bx,By,Cx,Cy,Dx,Dy)))
    return proper | col

In [ ]:
# 🔍 Verify solution
# Horizontal vs vertical — they cross
a = np.array([[[0.,1.],[2.,1.]]])
b = np.array([[[1.,0.],[1.,2.]]])
print("Cross (+):", segment_intersect(a, b))   # expect [[True]]

# Two parallel horizontal — no crossing
a2 = np.array([[[0.,0.],[3.,0.]], [[0.,1.],[3.,1.]]])
b2 = np.array([[[0.,2.],[3.,2.]]])
print("Parallel:", segment_intersect(a2, b2))  # expect [[False],[False]]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: perpendicular cross ────────────────────────────────────────────
a = np.array([[[0.,1.],[2.,1.]]]); b = np.array([[[1.,0.],[1.,2.]],[[3.,0.],[3.,2.]]])
r = segment_intersect(a, b)
assert r.shape==(1,2), f"Shape: {r.shape}"
assert r[0,0]==True and r[0,1]==False, f"Result: {r}"
print("Test 1 passed: perpendicular intersection and miss")

# ── Test 2: parallel segments ─────────────────────────────────────────────
a2 = np.array([[[0.,0.],[4.,0.]], [[0.,1.],[4.,1.]]])
b2 = np.array([[[0.,2.],[4.,2.]]])
r2 = segment_intersect(a2, b2)
assert not r2.any(), f"Parallel should not intersect: {r2}"
print("Test 2 passed: parallel no intersection")

# ── Test 3: T-intersection at endpoint ────────────────────────────────────
a3 = np.array([[[0.,0.],[4.,0.]]]); b3 = np.array([[[2.,0.],[2.,3.]]])
r3 = segment_intersect(a3, b3)
assert r3[0,0]==True, f"T-intersection: {r3}"
print("Test 3 passed: T-intersection at endpoint")

# ── Test 4: collinear overlapping ──────────────────────────────────────────
a4 = np.array([[[0.,0.],[3.,0.]]]); b4 = np.array([[[2.,0.],[5.,0.]]])
r4 = segment_intersect(a4, b4)
assert r4[0,0]==True, f"Collinear overlap: {r4}"
print("Test 4 passed: collinear overlapping")

# ── Test 5: large N=M=300 ─────────────────────────────────────────────────
rng=np.random.default_rng(7)
sa=rng.uniform(-10,10,(300,2,2)); sb=rng.uniform(-10,10,(300,2,2))
t0=time.time(); r5=segment_intersect(sa,sb); elapsed=time.time()-t0
assert r5.shape==(300,300), f"Shape: {r5.shape}"
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: N=M=300 ({elapsed:.3f}s)")

print("\nAll tests passed!")